# 00 · Viabilidad y fuentes

**Etapa 0.** Objetivo: comprobar con los datos reales si el proyecto es viable antes de plantear el análisis.

Pregunta de fondo: ¿existen, en los **mismos estudiantes argentinos**, el rendimiento, el nivel socioeconómico, el uso de IA y los pesos muestrales de PISA?

Este notebook solo audita. No contiene resultados sobre las hipótesis.


## 1. Preparación

Se localiza la raíz del proyecto subiendo carpetas hasta encontrar `README.md`, para no depender de rutas de ninguna computadora. Los datos viven en `data/raw/`, que no se sube a GitHub.

In [1]:
from pathlib import Path
import pandas as pd
import pyreadstat

raiz = Path.cwd()
while not (raiz / "README.md").exists():
    raiz = raiz.parent

ruta_2025 = raiz / "data" / "raw" / "pisa2025" / "extraido" / "CY09_MS_STU_PUF.sav"
ruta_codebook = raiz / "data" / "raw" / "pisa2025" / "docs" / "PISA2025_Codebook.xlsx"

print("Datos PISA 2025 presentes:", ruta_2025.exists())
print("Codebook presente:", ruta_codebook.exists())
print("pandas", pd.__version__, "| pyreadstat", pyreadstat.__version__)

Datos PISA 2025 presentes: True
Codebook presente: True
pandas 3.1.1 | pyreadstat 1.3.7


## 2. Matriz de fuentes

Estado de cada fuente según lo verificado hasta ahora. "No verificado" significa que no pudimos confirmarlo con una página oficial.

In [2]:
fuentes = pd.DataFrame([
    {"fuente": "OCDE PISA 2025, estudiantes", "rol": "Principal (Plan A)", "estado": "descargado y auditado",
     "uso de IA": "Sí (ST438Q01-04, IC170Q10, IC171Q10, AIUSESCH)", "rendimiento": "PV mate, lectura, ciencias",
     "pesos": "W_FSTUWT + 80 replicados", "terminos de uso": "no verificado"},
    {"fuente": "OCDE PISA 2022, estudiantes", "rol": "Respaldo / contexto", "estado": "descargado",
     "uso de IA": "No (según auditoría)", "rendimiento": "PV mate, lectura, ciencias",
     "pesos": "W_FSTUWT + 80 replicados", "terminos de uso": "no verificado"},
    {"fuente": "UNICEF Kids Online Argentina 2025", "rol": "Contexto descriptivo", "estado": "descargado (informe PDF)",
     "uso de IA": "Sí, agregado (sin microdatos)", "rendimiento": "No",
     "pesos": "No aplica", "terminos de uso": "Permiso escrito para reutilizar material"},
])
fuentes

,fuente,rol,estado,uso de IA,rendimiento,pesos,terminos de uso
0,"OCDE PISA 2025, estudiantes",Principal (Plan A),descargado y auditado,"Sí (ST438Q01-04, IC170Q10, IC171Q10, AIUSESCH)","PV mate, lectura, ciencias",W_FSTUWT + 80 replicados,no verificado
1,"OCDE PISA 2022, estudiantes",Respaldo / contexto,descargado,No (según auditoría),"PV mate, lectura, ciencias",W_FSTUWT + 80 replicados,no verificado
2,UNICEF Kids Online Argentina 2025,Contexto descriptivo,descargado (informe PDF),"Sí, agregado (sin microdatos)",No,No aplica,Permiso escrito para reutilizar material


## 3. Estructura del archivo de estudiantes de PISA 2025

Se leen primero solo los metadatos (nombres de columnas y número de filas), sin cargar los datos.

In [3]:
_, meta = pyreadstat.read_sav(ruta_2025, metadataonly=True)
columnas = meta.column_names

print("Filas:", meta.number_rows)
print("Columnas:", len(columnas))
print("Pesos replicados:", sum(c.startswith("W_FSTURWT") for c in columnas))
print("Primeros valores plausibles de matemática:", [c for c in columnas if c.startswith("PV") and c.endswith("MATH")][:3], "...")

Filas: 755721
Columnas: 885
Pesos replicados: 80
Primeros valores plausibles de matemática: ['PV1MATH', 'PV2MATH', 'PV3MATH'] ...


## 4. Variables clave en Argentina

Se cargan solo las columnas necesarias. Las preguntas de IA son de **frecuencia de uso de chatbots** para cuatro actividades escolares (resumir, investigar, redactar borradores, ayudar a aprender), con escala de 1 (nunca) a 5 (todos los días).

El archivo trae un `NaN` para cualquier faltante; todavía no distinguimos entre filtro del cuestionario, no administrado y no respuesta.

In [4]:
vars_ia = ["ST438Q01DA", "ST438Q02DA", "ST438Q03DA", "ST438Q04DA", "IC170Q10DA", "IC171Q10DA", "AIUSESCH"]
vars_resto = ["CNT", "CNTSCHID", "ESCS", "W_FSTUWT", "PV1MATH", "PV1READ", "PV1SCIE", "PV10MATH", "PV10READ", "PV10SCIE"]

datos, _ = pyreadstat.read_sav(ruta_2025, usecols=vars_resto + vars_ia)
argentina = datos.loc[datos["CNT"] == "ARG"].copy()

print("Estudiantes en el archivo:", len(datos), "| países:", datos["CNT"].nunique())
print("Estudiantes de Argentina:", len(argentina))
print("Escuelas de Argentina:", argentina["CNTSCHID"].nunique())

Estudiantes en el archivo: 755721 | países: 90
Estudiantes de Argentina: 11092
Escuelas de Argentina: 412


In [5]:
cobertura = (
    argentina[vars_resto[2:] + vars_ia]
    .notna()
    .mean()
    .mul(100)
    .round(1)
    .rename("% con dato")
    .to_frame()
)
cobertura

,% con dato
ESCS,88.4
W_FSTUWT,100.0
PV1MATH,100.0
PV1READ,100.0
PV1SCIE,100.0
PV10MATH,100.0
PV10READ,100.0
PV10SCIE,100.0
ST438Q01DA,70.8
ST438Q02DA,70.0


**Lectura.** Rendimiento y peso final están completos. El ESCS tiene dato para casi 9 de cada 10 estudiantes. Las cuatro preguntas de IA tienen dato para alrededor de 7 de cada 10, y las de TIC para la mitad. Esto es un hallazgo de cobertura, no una conclusión: antes de analizar hay que entender el origen de esos faltantes.

## 5. Distribución cruda de las cuatro preguntas de IA

Solo conteos (sin pesos) para ver si hay variación utilizable. **No son estimaciones para la población argentina.** La inferencia poblacional exige los pesos y la replicación, que se tratan en etapas posteriores.

In [6]:
etiquetas = {
    1: "1 Nunca o casi nunca", 2: "2 Una o dos veces al año", 3: "3 Una o dos veces al mes",
    4: "4 Una o dos veces por semana", 5: "5 Todos los días o casi",
}
preguntas = {
    "ST438Q01DA": "Resumir un texto",
    "ST438Q02DA": "Investigar un tema nuevo",
    "ST438Q03DA": "Redactar borradores",
    "ST438Q04DA": "Ayudar a aprender",
}

tabla = pd.DataFrame({
    texto: argentina[codigo].value_counts().sort_index()
    for codigo, texto in preguntas.items()
}).rename(index=etiquetas)
tabla.loc["Sin dato (NaN)"] = [argentina[c].isna().sum() for c in preguntas]
tabla

,Resumir un texto,Investigar un tema nuevo,Redactar borradores,Ayudar a aprender
1 Nunca o casi nunca,1786,1326,2375,1176
2 Una o dos veces al año,1163,1254,1222,1030
3 Una o dos veces al mes,1621,1902,1554,1427
4 Una o dos veces por semana,1821,1863,1375,1972
5 Todos los días o casi,1465,1421,1154,2047
Sin dato (NaN),3236,3326,3412,3440


## 6. Conclusión de viabilidad (propuesta, pendiente de aprobación)

- **Plan A es viable:** Argentina tiene rendimiento (valores plausibles), ESCS, pesos y preguntas de IA en los mismos estudiantes.
- **Los cuatro usos propuestos no coinciden con el instrumento.** PISA mide *frecuencia* de uso para cuatro actividades. "Copia directa" no se mide y no se puede reconstruir. Se analizará frecuencia por actividad, sin renombrarla como perfiles.
- **Riesgo principal:** alrededor de 3 de cada 10 estudiantes no tienen dato en las preguntas de IA. Hay que estudiar el origen antes de la Etapa 3.
- **Pendiente de verificar:** términos de uso de la OCDE, documentación técnica de 2025 y el factor de replicación.
- **Complementos:** PISA 2022 como contexto de rendimiento y desigualdad; UNICEF como contexto descriptivo, sin cruzarlo con PISA.

**Decisión que corresponde a la revisión:** aprobar el Plan A y pasar a la Etapa 1.